<a href="https://colab.research.google.com/github/ShahriarKamal2/Research-Practice/blob/experiment/experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, precision_score, recall_score, roc_curve
)

df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(f"Initial dataset shape: {df.shape}")
df.head()

Initial dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
df_clean = df.copy()

df_clean.drop(columns=["customerID"], inplace=True)

df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")
print(f"Missing values in TotalCharges before imputation: {df_clean['TotalCharges'].isna().sum()}")
df_clean["TotalCharges"] = df_clean["TotalCharges"].fillna(0.0)

service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]
for col in service_cols:
    df_clean[col] = df_clean[col].replace({"No internet service": "No"})

df_clean["MultipleLines"] = df_clean["MultipleLines"].replace({"No phone service": "No"})

df_clean["Churn"] = df_clean["Churn"].map({"No": 0, "Yes": 1})

print("Cleaned dataset info:")
print(df_clean["Churn"].value_counts(normalize=True).rename("Class Proportion"))

Missing values in TotalCharges before imputation: 11
Cleaned dataset info:
Churn
0    0.73463
1    0.26537
Name: Class Proportion, dtype: float64


In [3]:
df_clean["TenureCohort"] = pd.cut(
    df_clean["tenure"],
    bins=[-1, 12, 24, 48, 100],
    labels=["0-12m_New", "13-24m_Developing", "25-48m_Established", "49m+_Loyal"]
)
all_services = ["PhoneService", "MultipleLines"] + service_cols
df_clean["TotalServicesCount"] = (df_clean[all_services] == "Yes").sum(axis=1)

df_clean["ChargeDifference"] = df_clean["MonthlyCharges"] - (
    df_clean["TotalCharges"] / (df_clean["tenure"] + 1)
)

X = pd.get_dummies(df_clean.drop(columns=["Churn"]), drop_first=True)
y = df_clean["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "TotalServicesCount", "ChargeDifference"]
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(f"Processed X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

Processed X_train shape: (5634, 28), X_test shape: (1409, 28)
